In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# ==========================================
# 1. ADVANCED FEATURE ENGINEERING
# ==========================================
def create_elite_features(df):
    df = df.copy()
    # Logic remains identical to ensure parity
    df['Soil_Moisture_Low']  = (df['Soil_Moisture'] < 25).astype(int)
    df['Soil_Moisture_Med']  = df['Soil_Moisture'].between(25, 40).astype(int)
    df['Rainfall_Low']       = (df['Rainfall_mm'] < 1000).astype(int)
    df['Temp_High']          = (df['Temperature_C'] > 30).astype(int)
    df['High_Temp_Low_Rain'] = ((df['Temperature_C'] > 30) & (df['Rainfall_mm'] < 1000)).astype(int)
    df['Prev_Irrigation_Deficit'] = df['Previous_Irrigation_mm'] - (df['Soil_Moisture'] * 2)

    # Stress Signals (The 97% Boosters)
    df['VPD'] = df['Temperature_C'] * (1 - (df['Humidity'] / 100))
    df['Drought_Index'] = df['Temperature_C'] / (df['Rainfall_mm'] + 1)
    df['Evap_Potential'] = (df['Temperature_C'] * df['Sunlight_Hours']) / (df['Humidity'] + 1)
    
    return df

# ==========================================
# 2. GLOBAL THRESHOLD OPTIMIZER
# ==========================================
def optimize_threshold(oof_probs, y_true):
    best_bac = 0
    best_t = 0.5
    for t in np.linspace(0.01, 0.45, 200):
        preds = np.argmax(oof_probs, axis=1)
        preds[oof_probs[:, 2] >= t] = 2 
        score = balanced_accuracy_score(y_true, preds)
        if score > best_bac:
            best_bac = score
            best_t = t
    print(f"\n🏆 Peak CV Balanced Accuracy: {best_bac:.5f}")
    print(f"🎯 Winning Threshold: {best_t:.4f}")
    return best_t

# ==========================================
# 3. UPDATED ENSEMBLE CROSS-VALIDATION LOOP
# ==========================================
def train_ensemble(X, y, X_test, cat_features):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_probs_lgb = np.zeros((len(X), 3))
    oof_probs_xgb = np.zeros((len(X), 3))
    test_probs_lgb = np.zeros((len(X_test), 3))
    test_probs_xgb = np.zeros((len(X_test), 3))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"⚡ Processing Fold {fold+1}...")
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # --- LightGBM ---
        dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_features)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features, reference=dtrain)
        
        lgb_model = lgb.train(
            {'objective': 'multiclass', 'num_class': 3, 'learning_rate': 0.03, 'verbosity': -1},
            dtrain, num_boost_round=1500, valid_sets=[dval],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        oof_probs_lgb[val_idx] = lgb_model.predict(X_val)
        test_probs_lgb += lgb_model.predict(X_test) / 5

        # --- XGBoost (Updated API) ---
        xgb_model = xgb.XGBClassifier(
            n_estimators=1500, 
            learning_rate=0.03, 
            max_depth=6,
            tree_method='hist', 
            enable_categorical=True, 
            objective='multi:softprob',
            random_state=42,
            early_stopping_rounds=100  # Moved here from .fit()
        )
        
        # Fit with eval_set only
        xgb_model.fit(
            X_tr, y_tr, 
            eval_set=[(X_val, y_val)], 
            verbose=False
        )
        
        oof_probs_xgb[val_idx] = xgb_model.predict_proba(X_val)
        test_probs_xgb += xgb_model.predict_proba(X_test) / 5

    # Blending 50/50
    blended_oof = (oof_probs_lgb * 0.5) + (oof_probs_xgb * 0.5)
    blended_test = (test_probs_lgb * 0.5) + (test_probs_xgb * 0.5)
    
    return blended_oof, blended_test

# ==========================================
# 4. EXECUTION
# ==========================================

# 1. Feature Engineering & Categorical Casting
X = create_elite_features(train).drop(['id', 'Irrigation_Need'], axis=1)
y = train['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})
X_test = create_elite_features(test).drop(['id'], axis=1)

cat_features = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
                'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

for col in cat_features:
    unique_cats = X[col].unique()
    X[col] = pd.Categorical(X[col], categories=unique_cats)
    X_test[col] = pd.Categorical(X_test[col], categories=unique_cats)

# 2. Train Ensemble
print("--- 🚀 Training Modern Ensemble (LGBM + XGB) ---")
oof_probs, test_probs = train_ensemble(X, y, X_test, cat_features)

# 3. Optimize Threshold
winning_t = optimize_threshold(oof_probs, y)

# 4. Final Submission
test_preds = np.argmax(test_probs, axis=1)
test_preds[test_probs[:, 2] >= winning_t] = 2
sub['Irrigation_Need'] = pd.Series(test_preds).map({0: 'Low', 1: 'Medium', 2: 'High'})
sub.to_csv('submission_elite_ensemble_v3.csv', index=False)
print("✅ Done! File saved as submission_elite_ensemble_v3.csv")

the output of code  --- 🚀 Training Modern Ensemble (LGBM + XGB) ---
⚡ Processing Fold 1...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1355]	valid_0's multi_logloss: 0.0565533
⚡ Processing Fold 2...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1363]	valid_0's multi_logloss: 0.0568267
⚡ Processing Fold 3...
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1482]	valid_0's multi_logloss: 0.0573738
⚡ Processing Fold 4...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1202]	valid_0's multi_logloss: 0.0558955
⚡ Processing Fold 5...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1335]	valid_0's multi_logloss: 0.0559614

🏆 Peak CV Balanced Accuracy: 0.97218
🎯 Winning Threshold: 0.0719
✅ Done! File saved as submission_elite_ensemble_v3.csv
Error in callback <function _enable_matplotlib_integration.<locals>.configure_once at 0x00000231BE65B7E0> (for post_run_cell), with arguments args (<ExecutionResult object at 231d270f250, execution_count=21 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 231d41b8cd0, raw_cell="import numpy as np
import pandas as pd
import ligh.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/d%3A/FCI/03-Project/06-competitions/02-Predicting%20Irrigation%20Need/01-coding/Predicting-irrigation-Need.ipynb#X45sZmlsZQ%3D%3D> result=None>,),kwargs {}: